# 21 — Semantic Search & Retrieval

**Learning objective.** Build a retrieval pipeline and measure ranked relevance using vector similarity.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
docs=[
 'reset a forgotten password',
 'duplicate credit card transaction',
 'change email on account',
 'mobile application crashes on startup',
 'refund is missing from statement']
vec=TfidfVectorizer(ngram_range=(1,2),stop_words='english')
D=vec.fit_transform(docs)
def search(q,k=3):
    s=cosine_similarity(vec.transform([q]),D).ravel()
    order=np.argsort(-s)[:k]
    return pd.DataFrame({'rank':range(1,k+1),'document':[docs[i] for i in order],'score':np.round(s[order],3)})
search('card charged two times')

   rank                           document  score
0     1  duplicate credit card transaction  0.378
1     2         reset a forgotten password  0.000
2     3            change email on account  0.000

In [3]:
print('Exact lexical overlap can be brittle:')
print(search('cannot authenticate to profile',2).to_string(index=False))

Exact lexical overlap can be brittle:
 rank                          document  score
    1        reset a forgotten password    0.0
    2 duplicate credit card transaction    0.0


Dense sentence embeddings improve semantic matching for paraphrases. A production retriever should also consider ANN indexing, metadata filtering, reranking, freshness, access control and retrieval evaluation (`Recall@k`, `MRR`, `nDCG`).

---
    ## Production takeaways
    - Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
    - Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
    - Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Implement cosine-similarity retrieval
- Name retrieval-specific metrics and production constraints